In [1]:
cd ..

/Users/martinsevilla/Downloads/congestion-bogota


In [2]:
import os
import pandas as pd
import networkx as nx
import numpy as np
import pickle as pkl
import geopandas as gpd
from tqdm.notebook import tqdm

%load_ext autoreload
%autoreload 2

In [3]:
processed_data_dir = "data/datos_procesados"
roads_filename = 'data/2_Estructura_via/estructura_via_completa/MVI.geojson'

irregularities_files = [f"{processed_data_dir}/irregularidades_procesadas/" + f for f in os.listdir(f"{processed_data_dir}/irregularidades_procesadas/") if f.endswith('.parquet')]
graph_filename = f"{processed_data_dir}/conectivity_graph/road_graph.pickle"

# Download the world map data for visualization
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

# Load the base graph
with open(graph_filename, 'rb') as f:
    base_graph = pkl.load(f)
node_ids = list(base_graph.nodes())

# Load the roads GeoDataFrame
roads_gdf = gpd.read_file(roads_filename)
roads_gdf = roads_gdf.to_crs(epsg = 4326)
roads_gdf["id_mvi"] = roads_gdf["MVICIV"].astype(str) + "_" + roads_gdf["MVISVIA"].str.replace(" ", "")
roads_gdf['contador'] = roads_gdf.groupby('id_mvi').cumcount() + 1
roads_gdf['id_mvi'] = roads_gdf['id_mvi'] + '.' + roads_gdf['contador'].astype(str)
roads_gdf = (
    roads_gdf.explode(ignore_index=True)
       .dissolve(by='id_mvi')
       .reset_index()
)
roads_gdf['road_id'] = roads_gdf['id_mvi']
roads_gdf['buffer'] = roads_gdf.geometry.buffer(0.01)
roads_gdf.set_index('id_mvi', inplace=True)
road_geom = roads_gdf.geometry
del roads_gdf  # We don't need the full GeoDataFrame anymore

# Load all the irregularities data
use_cols = ["id_mvi", "timeframe", "trend", "speed", "regularSpeed"]
irregularities_df = pd.concat([pd.read_parquet(f)[use_cols] for f in irregularities_files], ignore_index=True)
# Compute the mean features for each road segment
mean_features = irregularities_df.groupby("id_mvi")[["trend", "regularSpeed"]].mean()

# set(mean_features.index.tolist()) < set(base_graph.nodes())

/var/folders/h9/yn87zkms09g2yrr0r9s65s6h0000gn/T/ipykernel_76770/948276180.py:28: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  roads_gdf['buffer'] = roads_gdf.geometry.buffer(0.01)


In [4]:
# Compute the disconnected nodes
components = [(len(c), base_graph.subgraph(c).copy()) for c in nx.connected_components(base_graph.to_undirected())]
components.sort(key=lambda x: x[0], reverse=True)
disconnected_nodes = set([node for component in components[1:] for node in component[1].nodes()])

## Plot disconnected nodes

In [5]:
m = road_geom.loc[list(disconnected_nodes)].explore(style_kwds={"weight":3}, height=900, width=1500)
m

## KNN Imputation

In [6]:
# Type of street for each street segment (node in the graph)
index = nx.get_node_attributes(base_graph, "MVITIPO").keys()
values = nx.get_node_attributes(base_graph, "MVITIPO").values()
street_types = (
    pd.DataFrame({"node_id": index, "street_type": values})
    .replace({" ": "Null"})
    .set_index("node_id")
)

k_hops = 10
# This dictionary will store the final results
# Keys: Node ID
# Values: A dictionary with the mean trend and regulgarSpeed of its neighbors
neighbor_features: dict[str, dict[str, float]] = {}
neighbor_nodes_with_data: dict[str, list[str]] = {}

# Iterate over each node in the graph
for node in tqdm(base_graph.nodes()):
    if node in disconnected_nodes:
        # If the node is disconnected, we skip it
        neighbor_features[node] = {"mean_trend": np.nan, "mean_regularSpeed": np.nan}
        continue
    k_hop_subgraph = nx.ego_graph(base_graph, node, radius=k_hops)
    neighbors = list(k_hop_subgraph.nodes())
    # Filter neighbors to only those that have the same stree type as the node
    node_street_type = street_types.loc[node, "street_type"]
    neighbors = [
        n
        for n in neighbors
        if street_types.loc[n, "street_type"] == node_street_type
        and n in mean_features.index
    ]
    neighbor_nodes_with_data[node] = neighbors
    # Check if the node has any neighbors
    if neighbors:
        neighbor_df = mean_features.loc[neighbors]

        neighbor_features[node] = {
            "mean_trend": neighbor_df["trend"].mean(),
            "mean_regularSpeed": neighbor_df["regularSpeed"].mean(),
        }
    else:
        # We assign NaN if the node has no neighbors with data
        neighbor_features[node] = {"mean_trend": np.nan, "mean_regularSpeed": np.nan}

neighbor_features_df = pd.DataFrame.from_dict(neighbor_features, orient="index")
print(
    "Proportion without imputable values:",
    neighbor_features_df.isna().sum().iloc[0] / len(neighbor_features_df),
)

  0%|          | 0/139107 [00:00<?, ?it/s]

Proportion without imputable values: 0.1306835745145823


In [5]:
# Type of street for each street segment (node in the graph)
index = nx.get_node_attributes(base_graph, "MVITIPO").keys()
values = nx.get_node_attributes(base_graph, "MVITIPO").values()
street_types = (
    pd.DataFrame({"node_id": index, "street_type": values})
    .replace({" ": "Null"})
    .set_index("node_id")
)

In [7]:
mean_features

,trend,regularSpeed
id_mvi,,
0_FT.1,0.054426,11.348287
0_FT.15,0.004392,16.490509
0_FT.21,-0.047801,18.407629
0_FT.9,-0.093625,23.419079
0_N.27,-0.054567,17.755008
...,...,...
9004746_FT.1,-0.099685,24.620590
9004746_FT.2,-0.141481,29.070947
9004747_FT.1,-0.095543,24.767705


In [10]:
mean_features_per_street_type = (
    mean_features.merge(street_types, how="left", left_index=True, right_index=True)
    .groupby("street_type")
    .mean()
)
mean_features_per_street_type

,trend,regularSpeed
street_type,,
AC,0.011113,19.088874
AK,-0.011490,19.289696
CL,0.095837,17.017156
DG,0.092757,19.048811
KR,0.074136,16.917991
Null,0.059190,20.483625
TV,0.039472,18.624655


In [40]:
neighbor_features_filled_df = neighbor_features_df.merge(street_types, how="left", left_index=True, right_index=True).merge(mean_features_per_street_type, how="left", left_on="street_type", right_index=True)
neighbor_features_filled_df["mean_trend"] = neighbor_features_filled_df["mean_trend"].fillna(neighbor_features_filled_df["trend"])
neighbor_features_filled_df["mean_regularSpeed"] = neighbor_features_filled_df["regularSpeed"].fillna(neighbor_features_filled_df["trend"])
neighbor_features_filled_df.drop(columns=["trend", "regularSpeed", "street_type"], inplace=True)
neighbor_features_filled_df

,mean_trend,mean_regularSpeed
0_B.1,-0.035576,17.017156
0_B.2,0.059190,20.483625
0_B.3,-0.463879,17.017156
0_B.4,0.406353,16.917991
0_B.5,0.756944,20.483625
...,...,...
9004787_B.1,0.413834,16.917991
91018094_N.1,-0.027425,18.624655
91022147_B.1,0.059190,20.483625
91022525_B.1,-0.119655,17.017156


In [13]:
# Export pickle files with the neighbor features and nodes with data
with open(os.path.join(processed_data_dir, "neighbor_features.pkl"), "rb") as f:
    neighbor_features_df = pkl.load(f)

In [7]:
# Export pickle files with the neighbor features and nodes with data
with open(f"neighbor_features.pickle", "wb") as f:
    pkl.dump(neighbor_features_df, f)
# Export the nodes with data
with open(f"neighbor_nodes_with_data.pickle", "wb") as f:
    pkl.dump(neighbor_nodes_with_data, f)

In [20]:
import random

main_node = random.choice(list(neighbor_nodes_with_data.keys()))
main_node_neighbors = neighbor_nodes_with_data[main_node]
m = road_geom.loc[[main_node]].explore(style_kwds={"weight": 7, "color": "green"}, height=900, width=1500)
road_geom.loc[main_node_neighbors].explore(m=m, style_kwds={"weight": 7, "color": "red"}, height=900, width=1500)
m